# Advanced ML / LLM Coding Practice

A second-stage interview notebook after the basic Python / NumPy / Pandas / PyTorch notebook.

## Topics
- sequence packing and packed masks
- MHA / MQA / GQA
- sliding-window + local/global attention
- linear attention
- RNN / LSTM / Conv1d
- InfoNCE
- classical ML mini-refresh
- activations + SwiGLU
- BatchNorm / LayerNorm / RMSNorm
- SFT masking
- greedy / temperature / top-k / top-p sampling
- KV cache
- KL + distillation
- online softmax
- gradient clipping, label smoothing, perplexity
- 3 hard integrated mocks

Every exercise has a TODO and a test immediately after it.

In [332]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
print(torch.__version__)

2.5.1+cu124


# 1. Sequence packing

## Exercise 1 — pack variable-length sequences into fixed blocks

In [ ]:
sequences = [[11,12,13],[21,22],[31,32,33,34],[41,42,43]]

def pack_sequences(sequences, block_size):
    # TODO: implement
    pass

def pack_sequences(sequences, block_size):
    # TODO: implement
    pass

In [334]:
tokens, seg = pack_sequences(sequences, 5)
assert tokens.shape == seg.shape == (2,5)
assert torch.equal(tokens[0], torch.tensor([11,12,13,21,22]))
assert torch.equal(seg[0], torch.tensor([0,0,0,1,1]))
print("✓")

✓


In [335]:
seg.shape

torch.Size([2, 5])

## Exercise 2 — packed causal mask

In [ ]:
def packed_causal_mask(segment_ids: torch.Tensor):
    # segment_ids: [T]
    # TODO: implement
    pass
    

In [ ]:
seg = torch.tensor([0,0,0,1,1])
m = None  # TODO: implement

In [338]:
seg = torch.tensor([0,0,0,1,1])
m = packed_causal_mask(seg)
expected = torch.tensor([
 [1,0,0,0,0],[1,1,0,0,0],[1,1,1,0,0],[0,0,0,1,0],[0,0,0,1,1]
], dtype=torch.bool)
assert torch.equal(m, expected)
print("✓")

tensor([[ True,  True,  True, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True, False, False],
        [False, False, False,  True,  True],
        [False, False, False,  True,  True]])
✓


# 2. MHA / MQA / GQA

## Exercise 3 — split/merge heads

In [ ]:
def split_heads(x: torch.Tensor, num_heads: int):
    # [B,T,D] -> [B,H,T,Dh]
    # TODO: implement
    pass

def merge_heads(x):
    # [B,H,T,Dh] -> [B,T,D]
    # TODO: implement
    pass

In [340]:
x=torch.randn(2,7,32)
h=split_heads(x,4)
assert h.shape==(2,4,7,8)
assert merge_heads(h).shape==(2,7,32)
print("✓")

✓


## Exercise 4 — MHA projections

In [ ]:
class MHAProjections(nn.Module):
    def __init__(self,d_model,num_heads):
        # TODO: implement
        pass
                
    def forward(self, x: torch.Tensor):
        # return q,k,v [B,H,T,Dh]
        # TODO: implement
        pass

In [342]:
m=MHAProjections(32,4)
q,k,v=m(torch.randn(2,6,32))
assert q.shape==k.shape==v.shape==(2,4,6,8)
print("✓")

✓


## Exercise 5 — Multi-Query Attention

In [ ]:
class MQAProjections(nn.Module):
    def __init__(self,d_model,num_q_heads):
        # TODO: implement
        pass
    def forward(self, x: torch.Tensor):
        # expand shared K/V across Q heads; return [B,Hq,T,Dh]
        # TODO: implement
        pass

In [344]:
m=MQAProjections(32,4)
q,k,v=m(torch.randn(2,5,32))
assert q.shape==k.shape==v.shape==(2,4,5,8)
assert torch.allclose(k[:,0],k[:,3])
assert torch.allclose(v[:,1],v[:,2])
print("✓")

torch.Size([2, 4, 5, 8])
torch.Size([2, 1, 5, 8])
✓


## Exercise 6 — Grouped-Query Attention

In [ ]:
class GQAProjections(nn.Module):
    def __init__(self,d_model,num_q_heads,num_kv_heads):
        # TODO: implement
        pass
    def forward(self,x):
        # Expand each KV head across its Q-head group.
        # Return q,k,v [B,Hq,T,Dh]
        # TODO: implement
        pass

In [ ]:
class GQA(nn.Module):
    def __init__(self,d_model,num_q_heads,num_kv_heads):
        # TODO: implement
        pass
    def forward(self,x):
        # Expand each KV head across its Q-head group.
        # Return q,k,v [B,Hq,T,Dh]
        # TODO: implement
        pass

In [347]:
m=GQAProjections(32,8,2)
q,k,v=m(torch.randn(2,5,32))
assert q.shape==k.shape==v.shape==(2,8,5,4)
assert torch.allclose(k[:,0],k[:,3])
assert not torch.allclose(k[:,0],k[:,4])
print("✓")

✓


# 3. Sliding-window and local + global attention

## Exercise 7 — sliding causal mask

In [ ]:
def sliding_causal_mask(T,window):
    # position i sees max(0,i-window+1)...i
    # TODO: implement
    pass

def sliding_causal_mask(T, window):
    # TODO: implement
    pass

In [ ]:
m = None  # TODO: implement

In [350]:
m=sliding_causal_mask(5,3)
exp=torch.tensor([[1,0,0,0,0],[1,1,0,0,0],[1,1,1,0,0],[0,1,1,1,0],[0,0,1,1,1]],dtype=torch.bool)
assert torch.equal(m,exp)
print("✓")

✓


## Exercise 8 — local + global causal mask

In [ ]:
def local_global_causal_mask(T, window, global_positions):
    # normal queries: local window + prior/equal global keys
    # global queries: all prior/equal positions
    # TODO: implement
    pass

In [352]:
m=local_global_causal_mask(6,2,[0,3])
assert m.shape==(6,6)
assert m[5,3] and m[3,:4].all() and m[2,0] and not m[5,1]
print("✓")

✓


# 4. Linear attention

Use the positive feature map `phi(x)=ELU(x)+1`.  
For non-causal linear attention, associate multiplication as `Q (K^T V)` rather than `(QK^T)V`.

In [ ]:
def phi(x):
    # TODO: implement
    pass

def linear_attention(Q,K,V,eps=1e-6):
    # Q,K,V [B,T,D]
    # TODO: numerator and normalization denominator
    # contract along T dimension
    # TODO: implement
    pass

In [354]:
Q,K,V=[torch.randn(2,7,8) for _ in range(3)]
o=linear_attention(Q,K,V)
assert o.shape==(2,7,8) and torch.isfinite(o).all()
assert (phi(torch.randn(10))>0).all()
print("✓")

✓


## Exercise 10 — causal linear attention with running state

In [ ]:
def causal_linear_attention(Q,K,V,eps=1e-6):
    # readable timestep loop is fine.
    # maintain cumulative K^T V and cumulative K.
    # TODO
    # TODO: implement
    pass

In [356]:
Q,K,V=[torch.randn(2,5,4) for _ in range(3)]
o=causal_linear_attention(Q,K,V)
assert o.shape==(2,5,4) and torch.isfinite(o).all()
print("✓")

✓


# 5. RNN / LSTM / CNN

## Exercise 11 — vanilla RNN cell

In [ ]:
class SimpleRNNCell(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        # TODO: implement
        pass
    def forward(self,x_t,h_prev):
        # h=tanh(Wx + Uh)
        # TODO: implement
        pass

In [358]:
c=SimpleRNNCell(6,10)
h=c(torch.randn(4,6),torch.randn(4,10))
assert h.shape==(4,10)
print("✓")

✓


## Exercise 12 — run RNN over a sequence

In [ ]:
def run_rnn(cell,x,h0=None):
    # x [B,T,D], return all h [B,T,H]
    # TODO: implement
    pass

In [360]:
x=torch.randn(3,7,6); c=SimpleRNNCell(6,10)
assert run_rnn(c,x).shape==(3,7,10)
print("✓")

✓


## Exercise 13 — LSTM cell

In [ ]:
class SimpleLSTMCell(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        # TODO: implement
        pass
    def forward(self,x_t,state):
        # TODO: implement
        pass

In [362]:
c=SimpleLSTMCell(6,10)
h,st=c(torch.randn(4,6),(torch.randn(4,10),torch.randn(4,10)))
assert h.shape==st.shape==(4,10)
print("✓")

✓


## Exercise 14 — Conv1d over a token sequence

In [ ]:
class SequenceConv(nn.Module):
    def __init__(self,d_in,d_out,kernel_size=3):
        # TODO: implement
        pass
        # TODO
    def forward(self,x):
        # input [B,T,D], Conv1d expects [B,C,T], return [B,T,Cout]
        # TODO: implement
        pass

In [364]:
assert SequenceConv(8,16,3)(torch.randn(2,10,8)).shape==(2,10,16)
print("✓")

✓


In [365]:
model = nn.Sequential(
    nn.Conv1d(4, 8, kernel_size=3, padding=1),
    nn.ReLU(),

    nn.Conv1d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),

    nn.Conv1d(16, 32, kernel_size=3, padding=1)
)

x = torch.randn(2, 4, 10)
y = model(x)

print(x.shape)
print(y.shape)

torch.Size([2, 4, 10])
torch.Size([2, 32, 10])


In [366]:
x = torch.tensor([
    [[1., 2., 3., 4., 5., 6.]]
])

pool = nn.AvgPool1d(
    kernel_size=2,
    stride=2
)
# Pooling usually makes the representation smaller while retaining coarse information.

y = pool(x)

print(x)
print(y)

tensor([[[1., 2., 3., 4., 5., 6.]]])
tensor([[[1.5000, 3.5000, 5.5000]]])


In [ ]:
model = nn.Sequential(
    nn.Conv1d(4, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AvgPool1d(kernel_size=2),

    nn.Conv1d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AvgPool1d(kernel_size=2)
)

x = torch.randn(2, 4, 16)

# TODO: implement training loop
pass

In [ ]:
x = torch.randn(
    2,   # batch
    3,   # RGB
    32,  # height
    32   # width
)

conv = nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    padding=1
)

y = None  # TODO: implement

print(x.shape)
print(y.shape)

In [ ]:
x = torch.randn(2, 3, 32, 32)

conv = nn.Conv2d(
    3,
    16,
    kernel_size=3,
    stride=2,
    padding=1
)

y = None  # TODO: implement

print(y.shape)

In [ ]:
x = torch.randn(
    2,   # batch
    3,   # channels
    8,   # depth / frames
    32,  # height
    32   # width
)

conv = nn.Conv3d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    padding=1
)

y = None  # TODO: implement

print(x.shape)
print(y.shape)

In [371]:
video = torch.randn(1, 3, 16, 64, 64)

conv = nn.Conv3d(
    3,
    32,
    kernel_size=(3, 3, 3),
    stride=(1, 2, 2),
    padding=1
)

out = conv(video)

print(video.shape)
print(out.shape)

torch.Size([1, 3, 16, 64, 64])
torch.Size([1, 32, 16, 32, 32])


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor):
        # TODO: implement
        pass

model = TinyCNN()

x = torch.randn(4, 3, 32, 32)
logits = model(x)

print(logits.shape)

# 6. InfoNCE

In [ ]:
def info_nce(a,b,temperature=0.1):
    # L2-normalize rows, similarity [B,B], diagonal positives
    # a, b: B, D

    # TODO: implement
    pass

def symmetric_info_nce(a,b,temperature=0.1):
    # average a->b and b->a
    # TODO: implement
    pass

In [374]:
a=torch.randn(8,16); b=a+0.05*torch.randn(8,16)
assert info_nce(a,b).item()<1.0
assert torch.isfinite(symmetric_info_nce(a,b))
print("✓")

✓


# 7. Classical ML mini-refresh

## Exercise 16 — ordinary least squares

In [ ]:
def ols(X,y):
    # w = pinv(X) @ y
    # Solves (X.T @ X) @ w = X.T @ y for w
    # TODO: implement
    pass

In [376]:
X=torch.tensor([[1.,0.],[1.,1.],[1.,2.],[1.,3.]])
y=torch.tensor([1.,3.,5.,7.])
assert torch.allclose(ols(X,y),torch.tensor([1.,2.]),atol=1e-5)
print("✓")

✓


## Exercise 17 — kNN

In [ ]:
def knn_predict(X_train,y_train,x_query,k=3):
    # Euclidean distance, majority vote
    # TODO: implement
    pass

In [378]:
X=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[10.,10.]])
y=torch.tensor([0,0,0,1])
assert knn_predict(X,y,torch.tensor([0.2,0.2]),3)==0
print("✓")

tensor([0, 1, 2])
tensor([0, 0, 0])
tensor(0)
✓


## Exercise 18 — one-feature decision stump

In [ ]:
def best_stump_threshold(x,y):
    # predict 1 iff x>threshold; search midpoint thresholds
    # return threshold, best_accuracy
    # TODO: implement
    pass

In [380]:
x=torch.tensor([1.,2.,3.,8.,9.,10.]); y=torch.tensor([0,0,0,1,1,1])
thr,acc=best_stump_threshold(x,y)
assert acc==1.0 and 3<=thr<8
print("✓")

✓


# 8. Activations and gated MLPs

In [ ]:
def my_sigmoid(x):
    # TODO: implement
    pass
def my_relu(x):
    # TODO: implement
    pass
def my_leaky_relu(x,alpha=0.01):
    # TODO: implement
    pass
def my_silu(x):
    # TODO: implement
    pass

In [382]:
x=torch.linspace(-3,3,20)
assert torch.allclose(my_sigmoid(x),torch.sigmoid(x),atol=1e-6)
assert torch.allclose(my_relu(x),F.relu(x))
assert torch.allclose(my_leaky_relu(x),F.leaky_relu(x,0.01))
assert torch.allclose(my_silu(x),F.silu(x),atol=1e-6)
print("✓")

✓


## Exercise 20 — SwiGLU

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,hidden_dim):
        # TODO: implement
        pass
        # TODO
    def forward(self,x):
        # TODO: implement
        pass

In [384]:
assert SwiGLU(32,64)(torch.randn(2,7,32)).shape==(2,7,32)
print("✓")

✓


# 9. BatchNorm / LayerNorm / RMSNorm

In [ ]:
class MyLayerNorm(nn.Module):
    def __init__(self,d,eps=1e-5):
        # TODO: implement
        pass
    def forward(self,x: torch.Tensor):
        # normalize final dimension
        # TODO: implement
        pass

In [386]:
x=torch.randn(4,7,16)
mine=MyLayerNorm(16); ref=nn.LayerNorm(16)
with torch.no_grad():
    ref.weight.copy_(mine.weight); ref.bias.copy_(mine.bias)
assert torch.allclose(mine(x),ref(x),atol=1e-5)
print("✓")

✓


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self,d,eps=1e-6):
        # TODO: implement
        pass
    def forward(self, x: torch.tensor):
        # do NOT subtract mean
        # TODO: implement
        pass

In [388]:
x=torch.randn(3,5,12); y=RMSNorm(12)(x)
assert y.shape==x.shape and torch.isfinite(y).all()
print("✓")

✓


In [389]:
def manual_batch_norm(x,eps=1e-5):
    # x [B,D]: feature-wise stats over batch
    # no running stats / affine needed
    # TODO
    pass

In [ ]:
class BatchNorm(nn.Module):
    def __init__(self, d, eps=1e-5, momentum=0.1):
        # TODO: implement
        pass

    def forward(self, x):
        # x: [B, D]

        # TODO: implement
        pass

In [391]:
x=torch.randn(64,8); y=manual_batch_norm(x)
assert torch.allclose(y.mean(0),torch.zeros(8),atol=1e-5)
assert torch.allclose(y.var(0,unbiased=False),torch.ones(8),atol=1e-4)
print("✓")

AttributeError: 'NoneType' object has no attribute 'mean'

# 10. SFT loss masking

In [ ]:
tokens=torch.tensor([[10,11,12,20,21,22,0],[13,14,30,31,0,0,0]])
roles=torch.tensor([[0,0,0,1,1,1,2],[0,0,1,1,2,2,2]])
# 0=user/prompt, 1=assistant, 2=padding

def make_sft_inputs_labels(tokens,roles,ignore_index=-100):
    # shifted input/target.
    # target contributes only if TARGET role is assistant.
    # TODO: implement
    pass

In [393]:
inp,lab=make_sft_inputs_labels(tokens,roles)
assert inp.shape==lab.shape==(2,6)
assert torch.equal(lab[0],torch.tensor([-100,-100,20,21,22,-100]))
print("✓")

tensor([[ True,  True, False, False, False,  True],
        [ True, False, False,  True,  True,  True]])
✓


# 11. Sampling

In [ ]:
def greedy_token(logits):
    # TODO: implement
    pass
def temperature_probs(logits,temperature):
    # TODO: implement
    pass


def top_k_filter(logits,k):
    # keep k largest logits, others -inf
    # TODO: implement
    pass

def top_k_filter(logits, k):
    # TODO: implement
    pass

In [395]:
x=torch.tensor([[1.,5.,3.,2.]])
assert greedy_token(x).item()==1
assert temperature_probs(x,2.0).max()<temperature_probs(x,1.0).max()
y=top_k_filter(x,2)
assert torch.isfinite(y).sum()==2
print("✓")

✓


## Exercise 26 — top-p / nucleus filtering

In [ ]:
def top_p_filter(logits,p):
    # sort logits descending, compute softmax + cumulative probs,
    # retain minimal prefix crossing p, scatter mask back.
    # 1. Sort logits from largest to smallest
    # TODO: implement
    pass
    

In [397]:
# filtered = top_p_filter(logits, p=0.9)

# probs = torch.softmax(filtered, dim=-1)

# token = torch.multinomial(probs, num_samples=1)

In [398]:
x=torch.tensor([[5.,4.,1.,0.]])
y=top_p_filter(x,0.8)
assert y.shape==x.shape and torch.isfinite(y[0,0])
assert torch.isfinite(y).sum()>=1
print("✓")

✓


# 12. KV cache

In [ ]:
def append_kv_cache(k_cache,v_cache,k_new,v_new):
    # sequence dimension is -2 for [B,H,T,Dh]
    # TODO
    # TODO: implement
    pass

In [400]:
kc=torch.randn(2,4,5,8); vc=torch.randn(2,4,5,8)
kn=torch.randn(2,4,1,8); vn=torch.randn(2,4,1,8)
k2,v2=append_kv_cache(kc,vc,kn,vn)
assert k2.shape==v2.shape==(2,4,6,8)
assert torch.allclose(k2[:,:,:5],kc)
print("✓")

✓


In [ ]:
def cached_attention(q,k_cache,v_cache):
    # q [B,H,1,Dh], cache [B,H,T,Dh]
    # TODO: implement
    pass

In [402]:
q=torch.randn(2,4,1,8); k=torch.randn(2,4,6,8); v=torch.randn(2,4,6,8)
assert cached_attention(q,k,v).shape==(2,4,1,8)
print("✓")

✓


# 13. KL / distillation

In [ ]:
def distill_kl(student_logits,teacher_logits,temperature=1.0):
    # mean over batch of sum p_teacher*(log p_teacher-log p_student)
    # TODO: implement
    pass

def temperature_distill_loss(student_logits,teacher_logits,T=2.0):
    # KL at T, multiply by T^2
    # mean over batch of sum p_teacher*(log p_teacher-log p_student)
    # TODO: implement
    pass

def mixed_distillation_loss(student_logits,teacher_logits,labels,alpha=0.5,T=2.0):
    # alpha*hard CE + (1-alpha)*temperature distillation
    # TODO: implement
    pass

In [404]:
s=torch.tensor([[1.,2.,3.],[2.,1.,0.]])
assert abs(distill_kl(s,s).item())<1e-6
assert torch.isfinite(temperature_distill_loss(torch.randn(4,10),torch.randn(4,10)))
assert torch.isfinite(mixed_distillation_loss(torch.randn(8,5),torch.randn(8,5),torch.randint(0,5,(8,))))
print("✓")

✓


# 14. Stable + online softmax

In [405]:
def stable_softmax(x,dim=-1):
    # TODO
    pass

In [406]:
x=torch.tensor([[1000.,1001.,1002.]])
p=stable_softmax(x)
assert torch.isfinite(p).all() and torch.allclose(p.sum(-1),torch.ones(1))
print("✓")

TypeError: isfinite(): argument 'input' (position 1) must be Tensor, not NoneType

## Exercise 31 — online softmax statistics

Maintain running maximum `m` and denominator `l`, rescaling the old denominator whenever a new maximum arrives.

In [ ]:
def online_softmax_stats(x):
    # x [T], return m,l with l=sum(exp(x-m))
    # TODO: implement
    pass

In [408]:
x=torch.tensor([1000.,1002.,999.,1003.])
m,l=online_softmax_stats(x)
assert torch.allclose(m,x.max())
assert torch.allclose(l,torch.exp(x-x.max()).sum(),atol=1e-6)
print("✓")

✓


## Exercise 32 — blockwise online-softmax statistics

In [ ]:
def blockwise_softmax_stats(x,block_size):
    # merge each block's max/denominator into running stats
    # TODO: implement
    pass

In [410]:
x=torch.randn(37)*20
m,l=blockwise_softmax_stats(x,7)
assert torch.allclose(m,x.max())
assert torch.allclose(l,torch.exp(x-x.max()).sum(),atol=1e-5)
print("✓")

✓


# 15. Extra high-value pieces

## Exercise 33 — gradient clipping

In [ ]:
model=nn.Linear(8,2); opt=torch.optim.SGD(model.parameters(),lr=.1)
x=torch.randn(32,8)*100; y=torch.randint(0,2,(32,))
opt.zero_grad()
logits = model(x)
loss = nn.functional.cross_entropy(logits, y)
loss.backward()
pre_clip_norm = None  # TODO: implement
opt.step()
# TODO: one training step and clip global grad norm to 1.0.
# Save returned pre-clip norm in pre_clip_norm.
# pre_clip_norm=None

In [412]:
x.shape, y.shape

(torch.Size([32, 8]), torch.Size([32]))

In [413]:
assert pre_clip_norm is not None
assert torch.isfinite(torch.as_tensor(pre_clip_norm))
print("✓")

✓


In [414]:
# with torch.amp.autocast(device_type=device, dtype=torch.float16):


## Exercise 34 — label smoothing

In [ ]:
def smooth_labels(labels,num_classes,eps=.1):
    # correct class 1-eps, distribute eps over incorrect classes
    # TODO: implement
    pass

In [416]:
q=smooth_labels(torch.tensor([0,2]),4,.1)
assert q.shape==(2,4)
assert torch.allclose(q.sum(-1),torch.ones(2))
assert torch.allclose(q[0,0],torch.tensor(.9))
print("✓")

✓


In [417]:
q

tensor([[0.9000, 0.0333, 0.0333, 0.0333],
        [0.0333, 0.0333, 0.9000, 0.0333]])

## Exercise 35 — masked mean + perplexity

In [ ]:
def masked_mean(x,mask):
    # x [B,T,D], mask [B,T]
    # return [B,D]
    # TODO: implement
    pass
def perplexity(avg_nll):
    # TODO: implement
    pass

In [419]:
x=torch.tensor([[[1.],[3.],[100.]],[[2.],[4.],[6.]]])
mask=torch.tensor([[1,1,0],[1,1,1]],dtype=torch.bool)
assert torch.allclose(masked_mean(x,mask).squeeze(-1),torch.tensor([2.,4.]))
assert abs(perplexity(torch.tensor(math.log(10.))).item()-10)<1e-5
print("✓")

torch.Size([2, 3, 1]) torch.Size([2, 3, 1])
✓


In [420]:
import torch
import torch.nn as nn

# Fake data
X = torch.randn(100, 4)              # 100 examples, 4 features
y = torch.randint(0, 2, (100, 1)).float()

# Model outputs ONE logit
model = nn.Linear(4, 1)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(100):
    # Forward
    logits = model(X)                 # [100, 1]

    # Loss
    loss = loss_fn(logits, y)

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(loss.item())

0.6616514325141907


# 16. Hard integrated mocks

## Hard Mock 1 — Compact decoder block (60 min)

Given padded integer sequences, `vocab_size`, `d_model`, `num_q_heads`, `num_kv_heads`:

1. next-token shift
2. assistant-only SFT mask
3. embeddings
4. RMSNorm
5. GQA
6. RoPE
7. causal + padding mask
8. SwiGLU
9. residual connections
10. LM head
11. token-masked CE
12. 20 training steps
13. token-average loss
14. gradient clipping
15. greedy generation with KV cache

Twists:
- derive KV-cache memory for MHA vs GQA vs MQA
- explain RoPE placement
- explain loss mask vs attention mask
- support sequence lengths longer than the initial cache

In [421]:
import math
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader

torch.manual_seed(0)


PAD_ID = 0
IGNORE_INDEX = -100
vocab_size = 64
d_model = 64
num_q_heads = 8
num_kv_heads = 2
max_seq_len = 16

batch_size = 4
num_steps = 20

tokens = torch.tensor([
    [1,  5,  8,  4,  9,  2,  0,  0],
    [3,  7,  2, 11,  6,  4, 12,  2],
    [4, 10, 12,  3,  8,  1,  0,  0],
    [9,  2,  5,  7,  4,  6,  3,  2],
    [8,  1,  3,  5,  2, 10,  0,  0],
    [2,  4,  6,  8, 10, 12, 14,  2],
    [5,  3,  9,  1,  7, 11,  0,  0],
    [6,  8,  2,  4,  1,  9,  5,  2],
], dtype=torch.long)

roles = torch.tensor([
    [0,0,0,1,1,1,2,2],
    [0,0,0,1,1,1,1,1],
    [0,0,1,1,1,1,2,2],
    [0,0,0,1,1,1,1,1],
    [0,0,1,1,1,1,2,2],
    [0,0,0,1,1,1,1,1],
    [0,0,1,1,1,1,2,2],
    [0,0,0,1,1,1,1,1],
], dtype=torch.long)

In [ ]:
def make_sft_batch(
        tokens: torch.Tensor, # [B, T]
        roles: torch.Tensor # [B, T]
    ) -> tuple[torch.Tensor, torch.Tensor]:
    
    # TODO: implement
    pass
    

In [423]:
inputs, labels = make_sft_batch(tokens, roles)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        # TODO: implement
        pass
    def forward(self, x: torch.Tensor): # [B, T, D]
        # TODO: implement
        pass

In [ ]:
def split_heads(x: torch.Tensor, num_heads: int):
    # B, T, D -> B, H, T, Dh
    # TODO: implement
    pass

In [ ]:
def merge_heads(x: torch.Tensor):
    # B, H, T, Dh -> B, T, D
    # TODO: implement
    pass

In [ ]:
class RoPE(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int, rope_base_theta: int = 10000):
        # TODO: implement
        pass

    def _build_cache(self, seq_len):
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor, position_ids: torch.Tensor):
        # B, H, T, Dh
        # B, T
        # TODO: implement
        pass

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self, 
        d_model: int, 
        num_q_heads: int,
        num_kv_heads: int,
        max_seq_len: int
    ):
        # TODO: implement
        pass

    def forward(self, 
        x: torch.Tensor,
        attention_mask: torch.Tensor = None,
        past_k: torch.Tensor = None,
        past_v: torch.Tensor = None,
        use_cache: bool = False,
        position_ids: torch.Tensor = None
    ):
        # TODO: implement
        pass
        

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim):
        # TODO: implement
        pass


    def forward(self, x: torch.Tensor):

        # TODO: implement
        pass

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        num_q_heads,
        num_kv_heads,
        mlp_hidden_dim,
        max_seq_len=16,
    ):
        # TODO: implement
        pass

    def forward(
        self,
        x: torch.Tensor,
        attention_mask=None,
        past_k=None,
        past_v=None,
        use_cache=False,
        position_ids: torch.Tensor =None,
    ):
        # TODO: implement
        pass

In [ ]:
class TinyDecoderLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_q_heads,
        num_kv_heads,
        mlp_hidden_dim,
        max_seq_len=16,
    ):
        # TODO: implement
        pass

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_k=None,
        past_v=None,
        use_cache=False,
        position_ids: torch.Tensor= None,
    ):
        # TODO: implement
        pass
        

In [ ]:
def make_attention_mask(input_ids: torch.Tensor, pad_id: int = PAD_ID):
    """
    Return boolean mask suitable for attention.

    Requirements:
    - causal
    - padded KEYS should not be attended to

    Expected conceptual shape:
        [B, 1, T, T]
    """
    # TODO: implement
    pass

In [ ]:
dataset = None  # TODO: implement
dataloader = None  # TODO: implement

model = None  # TODO: implement

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-3
)

In [ ]:
model.train()
step = 0

total_loss = 0
total_count = 0

# TODO: implement training loop
pass

In [ ]:
@torch.no_grad()
def greedy_generate(
    model,
    prompt_ids,
    max_new_tokens,
    eos_id=None,
):
    """
    prompt_ids: [B,T]

    Requirements:
    - run prompt once
    - build KV cache
    - afterward feed only ONE new token at a time
    - use absolute positions correctly for RoPE
    - grow beyond initial RoPE cache if needed
    """

    # TODO: implement
    pass

In [436]:
# Simple generation smoke test

prompt = torch.tensor([
    [1, 5, 8]
], dtype=torch.long)

generated = greedy_generate(
    model,
    prompt_ids=prompt,
    max_new_tokens=10,
    eos_id=None,
)

print("Prompt:")
print(prompt)

print("\nGenerated token IDs:")
print(generated)

print("\nGenerated shape:")
print(generated.shape)

assert generated.shape[0] == prompt.shape[0]
assert generated.shape[1] >= prompt.shape[1]
assert torch.equal(generated[:, :prompt.shape[1]], prompt)

print("\n✓ greedy_generate runs successfully")

torch.Size([1, 64])
Prompt:
tensor([[1, 5, 8]])

Generated token IDs:
tensor([[ 1,  5,  8,  4,  9,  2, 10, 12, 14,  2, 10, 12, 14]])

Generated shape:
torch.Size([1, 13])

✓ greedy_generate runs successfully


In [ ]:
def kv_cache_elements(
    batch_size,
    seq_len,
    head_dim,
    num_kv_heads,
):
    """
    Return number of scalar elements in the K+V cache.

    Compute for:
    - MHA
    - GQA
    - MQA
    """
    # TODO: implement
    pass

## Hard Mock 2 — Long-context attention lab (60 min)

Given hidden states `[B,T,D]`:

A. standard causal attention  
B. sliding-window attention  
C. add global tokens  
D. causal linear attention with recurrent sufficient statistics  
E. calculate the main intermediate tensor/state shapes and asymptotic memory

Twists:
- identify which versions preserve exact softmax attention
- explain what simple kernelized linear attention changes
- reason about receptive field after stacking several local-attention layers
- compare prefill vs autoregressive decode costs

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

B = 2
T = 16
D = 32

window_size = 4
global_positions = [0, 7]

x = None  # TODO: implement

In [ ]:
class StandardCausalAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.o = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: [B,T,D]

        Return:
            output: [B,T,D]
            attn_probs: [B,T,T]
        """

        # TODO: implement
        pass

In [440]:
m = StandardCausalAttention(D)

out, probs = m(x)

assert out.shape == (B,T,D)
assert probs.shape == (B,T,T)

upper = torch.triu(probs, diagonal=1)
assert torch.allclose(upper, torch.zeros_like(upper), atol=1e-6)

print("✓ standard causal attention")

✓ standard causal attention


In [ ]:
def make_sliding_causal_mask(T, window_size, device=None):
    """
    Position i may attend only to:
        max(0, i-window_size+1) ... i

    Return:
        bool [T,T]
    """
    # TODO: implement
    pass

In [442]:
mask = make_sliding_causal_mask(
    T=6,
    window_size=3
)

expected = torch.tensor([
    [1,0,0,0,0,0],
    [1,1,0,0,0,0],
    [1,1,1,0,0,0],
    [0,1,1,1,0,0],
    [0,0,1,1,1,0],
    [0,0,0,1,1,1],
], dtype=torch.bool)

assert torch.equal(mask.cpu(), expected)

print("✓ sliding mask")

✓ sliding mask


In [ ]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, d_model, window_size):
        super().__init__()

        self.window_size = window_size
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.o = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: [B,T,D]
        """

        # TODO: implement
        
        pass

In [ ]:
def make_local_global_mask(
    T,
    window_size,
    global_positions,
    device=None
):
    """
    Return bool mask [T,T].
    """

    # TODO: implement
    
    pass
    

In [445]:
mask = make_local_global_mask(
    T=10,
    window_size=3,
    global_positions=[0,5]
)
print(mask)

assert mask.shape == (10,10)

# global token 5 sees all earlier positions
assert mask[5, :6].all()

# token 9 can see global token 5
assert mask[9,5]

# token 9 should not see old non-global token 2
assert not mask[9,2]

# causality
assert not mask[4,7]

print("✓ local + global mask")

tensor([[ True, False, False, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False, False, False],
        [ True, False,  True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True, False, False, False, False],
        [ True, False, False, False,  True,  True,  True, False, False, False],
        [ True, False, False, False, False,  True,  True,  True, False, False],
        [ True, False, False, False, False,  True,  True,  True,  True, False],
        [ True, False, False, False, False,  True, False,  True,  True,  True]])
✓ local + global mask


In [ ]:
class LocalGlobalAttention(nn.Module):
    def __init__(
        self,
        d_model,
        window_size,
        global_positions
    ):
        # TODO: implement
        pass

    def forward(self, x):

        # TODO: implement
        pass

In [ ]:
def phi(x):
    # TODO: implement
    pass

In [ ]:
class CausalLinearAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.o = nn.Linear(d_model, d_model)

    def forward(self, x, eps=1e-6):
        """
        x: [B,T,D]

        Do NOT construct [B,T,T].

        Return:
            output [B,T,D]
        """


        # TODO: implement
        
        pass

In [449]:
m = CausalLinearAttention(D)

out = m(x)
print(out.shape)
assert out.shape == (B,T,D)
assert torch.isfinite(out).all()

print("✓ causal linear attention")

torch.Size([2, 16, 32])
✓ causal linear attention


In [ ]:
def attention_complexity(
    B,
    T,
    D,
    window_size
):
    """
    Return rough main intermediate sizes.

    Ignore constant factors where appropriate.
    """

    # TODO: implement
    
    pass

In [ ]:
stats = None  # TODO: implement

print(stats)

In [ ]:
def receptive_field(window_size, num_layers):
    """
    Assume causal sliding-window attention.

    Roughly how many previous positions can influence
    one token after num_layers?
    """
    # Each layer extends the lookback capacity by (window_size - 1)
    # TODO: implement
    
    pass

print(receptive_field(window_size=3, num_layers=4))

In [453]:
# STANDARD ATTENTION -> O(T^2D) and O(Td) during inference
# LOCAL ATTENTION -> O(T*window_size*d) and O(window_size*d) during inference
# LINEAR ATTENTION -> O(TD^2) fixed during training and inference O(D^2)

In [454]:
standard = StandardCausalAttention(D)
sliding = SlidingWindowAttention(D, window_size=4)
local_global = LocalGlobalAttention(
    D,
    window_size=4,
    global_positions=[0,7]
)
linear = CausalLinearAttention(D)

y1, p1 = standard(x)
y2, p2 = sliding(x)
y3, p3 = local_global(x)
y4 = linear(x)
assert y1.shape == y2.shape == y3.shape == y4.shape == x.shape

print("✓ all attention variants run")

✓ all attention variants run


## Hard Mock 3 — Multi-objective student training (60 min)

Inputs:
- teacher logits `[B,C]`
- student inputs `[B,Din]`
- paired teacher features `[B,E]`
- labels `[B]`

Build a student with:
- classifier head
- embedding/projection head

Optimize:
`alpha * CE + beta * T^2*KL + gamma * symmetric_InfoNCE`

Requirements:
- correct teacher detachment
- temperature scaling
- per-loss logging
- gradient clipping
- eval/no_grad
- accuracy + macro-F1
- ablation flags for every objective

Twists:
- teacher/student class vocabularies are accidentally permuted
- one modality has duplicate positives in the batch
- validation loss improves while task metric degrades

In [246]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)

N = 512
Din = 24
E = 32
C = 6

batch_size = 32
num_epochs = 15

alpha = 1.0   # CE
beta = 1.0    # KL distillation
gamma = 0.2   # InfoNCE

temperature = 2.0
contrastive_temperature = 0.1

USE_CE = True
USE_KL = True
USE_INFONCE = True


student_inputs = torch.randn(N, Din)

labels = torch.randint(
    low=0,
    high=C,
    size=(N,),
    dtype=torch.long
)

teacher_logits = torch.randn(N, C)

teacher_logits[
    torch.arange(N),
    labels
] += 3.0

teacher_features = torch.randn(N, E)

# Make teacher features somewhat structured by class
class_centers = torch.randn(C, E)

teacher_features = (
    teacher_features * 0.3
    + class_centers[labels]
)


train_n = int(0.8 * N)

train_inputs = student_inputs[:train_n]
val_inputs = student_inputs[train_n:]

train_labels = labels[:train_n]
val_labels = labels[train_n:]

train_teacher_logits = teacher_logits[:train_n]
val_teacher_logits = teacher_logits[train_n:]

train_teacher_features = teacher_features[:train_n]
val_teacher_features = teacher_features[train_n:]

train_ds = TensorDataset(
    train_inputs,
    train_labels,
    train_teacher_logits,
    train_teacher_features
)

val_ds = TensorDataset(
    val_inputs,
    val_labels,
    val_teacher_logits,
    val_teacher_features
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
class StudentModel(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_classes,
        embedding_dim
    ):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        self.classifier_head = nn.Linear(hidden_dim, num_classes)
        self.projection_head = nn.Linear(hidden_dim, embedding_dim)

        # TODO:
        # classifier head -> [B,C]

        # TODO:
        # projection / embedding head -> [B,E]

    def forward(self, x):
        """
        Return:
            logits:   [B,C]
            features: [B,E]
        """
        # TODO: implement
       
        pass

In [296]:
model = StudentModel(
    input_dim=Din,
    hidden_dim=64,
    num_classes=C,
    embedding_dim=E
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

logits, features = model(
    torch.randn(8, Din)
)

assert logits.shape == (8, C)
assert features.shape == (8, E)

print("✓ model shapes")

✓ model shapes


In [ ]:
def distillation_loss(
    student_logits,
    teacher_logits,
    temperature
):
    """
    student_logits: [B,C]
    teacher_logits: [B,C]

    Teacher should not receive gradients.
    """

    # TODO: implement
    
    pass

In [298]:
x = torch.randn(8, C)

loss = distillation_loss(
    x,
    x.clone(),
    temperature=2.0
)

assert abs(loss.item()) < 1e-5

print("✓ KL loss")

✓ KL loss


In [ ]:
def info_nce(
    student_features,
    teacher_features,
    temperature=0.1
):
    """
    Assume matching pairs are on the diagonal.

    student_features: [B,E]
    teacher_features: [B,E]
    """

    # TODO:
    # normalize student features
    
    pass

In [ ]:
def symmetric_info_nce(
    student_features,
    teacher_features,
    temperature=0.1
):

    # TODO: implement
    pass

In [301]:
f = torch.randn(16, E)

loss = symmetric_info_nce(
    f,
    f + 0.01 * torch.randn_like(f),
    temperature=0.1
)

assert torch.isfinite(loss)

print("✓ InfoNCE")

✓ InfoNCE


In [ ]:
def compute_losses(
    student_logits,
    student_features,
    labels,
    teacher_logits,
    teacher_features,
    alpha=1.0,
    beta=1.0,
    gamma=0.2,
    temperature=2.0,
    contrastive_temperature=0.1,
    use_ce=True,
    use_kl=True,
    use_infonce=True,
):
    """
    Return:
        total_loss: torch.Tensor
        losses_dict: dict containing individual losses (as floats or tensors)
    """
    # TODO: implement
    
    pass

In [ ]:
def macro_f1(
    y_true,
    y_pred,
    num_classes
):
    # TODO: implement
    pass

In [304]:
yt = torch.tensor([0,0,1,1,2,2])
yp = torch.tensor([0,0,1,2,2,2])

f1 = macro_f1(
    yt,
    yp,
    num_classes=3
)

assert 0 <= f1 <= 1

print("macro F1:", f1)

macro F1: 0.8222222269905938


In [ ]:
# TODO: implement training loop
pass

In [ ]:
model.eval()

all_preds = []
all_targets = []

val_total = 0.0
val_count = 0

with torch.no_grad():

    for (
        xb,
        yb,
        teacher_logits_b,
        teacher_features_b
    ) in val_loader:

        # Forward
        student_logits_b, student_features_b = model(xb)

        # Losses
        total_loss, _ = compute_losses(
            student_logits=student_logits_b,
            student_features=student_features_b,
            labels=yb,
            teacher_logits=teacher_logits_b,
            teacher_features=teacher_features_b,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
            temperature=temperature,
            contrastive_temperature=contrastive_temperature,
            use_ce=USE_CE,
            use_kl=USE_KL,
            use_infonce=USE_INFONCE,
        )

        # Predictions
        preds = torch.argmax(student_logits_b, dim=-1)

        # Collect predictions/targets
        all_preds.append(preds)
        all_targets.append(yb)

        # Accumulate validation loss
        batch_n = xb.size(0)
        val_total += total_loss.item() * batch_n
        val_count += batch_n

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

accuracy = (
    all_preds == all_targets
).float().mean().item()

f1 = None  # TODO: implement

avg_val_loss = val_total / val_count

print("validation loss:", avg_val_loss)
print("accuracy:", accuracy)
print("macro F1:", f1)

In [ ]:
teacher_classes = [
    "cat",
    "dog",
    "car",
    "tree",
    "bird",
    "plane"
]
student_classes = [
    "dog",
    "cat",
    "car",
    "bird",
    "tree",
    "plane"
]

def align_teacher_logits(
    teacher_logits,
    teacher_classes,
    student_classes
):
    """
    Return teacher logits reordered so dimension c
    means the same class as student dimension c.
    """

    # TODO: implement
   
    pass

In [294]:
teacher_classes = ["a","b","c"]
student_classes = ["b","c","a"]

logits = torch.tensor([
    [10.,20.,30.]
])

aligned = align_teacher_logits(
    logits,
    teacher_classes,
    student_classes
)

assert torch.equal(
    aligned,
    torch.tensor([[20.,30.,10.]])
)

print("✓ class alignment")

✓ class alignment


In [ ]:
def info_nce(
    student_features,
    teacher_features,
    temperature=0.1
):
    """
    Assume matching pairs are on the diagonal.

    student_features: [B,E]
    teacher_features: [B,E]
    """

    # TODO:
    # normalize student features
    
    pass

In [ ]:
def multi_positive_info_nce(
    student_features,
    teacher_features,
    pair_ids,
    temperature=0.1
):
    """
    Any teacher example with the same pair_id
    should count as a positive.

    Hint:
    positive_mask[i,j] =
        pair_ids[i] == pair_ids[j]
    """
    # TODO: implement
   
    pass

# 17. Final readiness checklist

Be able to implement or explain:
- packing + packed masks
- MHA/MQA/GQA and their KV-cache implications
- causal/padding/sliding/global masks
- learned/sinusoidal/RoPE positional methods
- standard vs linear attention
- RNN/LSTM/Conv1d
- InfoNCE
- OLS/kNN/decision-stump basics
- ReLU/GELU/SiLU/SwiGLU
- BatchNorm/LayerNorm/RMSNorm
- SFT response-only loss masking
- temperature/top-k/top-p
- KV caching
- KL/distillation
- stable/online softmax
- gradient clipping/label smoothing/perplexity